# Metrics and baselines

In this notebook, we walk through measuring error of algorithm outputs with respect to statistical baselines.

## Simulate instances

We start by creating a local workspace and adding three simulated instances to it. Each instance has 1,000 samples in three dimensions from a translated Gaussian, clipped to a ball. [simulate_clipped_normal](https://saminulh.github.io/dpstats/library/reference/simulations/dpstats.simulations.mean_estimation.simulate_clipped_normal/) also returns the metric arguments for each instance, including:

1. `sample_mean`, the mean of the 1,000 samples.
2. `sample_mean_l2_se`, the standard error of the sample mean.

In [ ]:
from pathlib import Path

from dpstats import init_workspace, open_collection
from dpstats.simulations.mean_estimation import simulate_clipped_normal
from dpstats.tasks import MeanEstimation

workspace = init_workspace(Path("metrics-workspace"))
collection = open_collection(workspace, MeanEstimation)
instances = []
for seed in (0, 1, 2):
    metadata, input, arguments = simulate_clipped_normal(
        seed=seed, n_samples=500, dimension=10
    )
    instances.append(
        collection.add_instance(
            metadata, input, arguments, label=f"clipped normal, seed {seed}"
        )
    )
print("sample mean:", arguments["sample_mean"])
print("standard error of the sample mean:", arguments["sample_mean_l2_se"])

## Run a sweep

Next, we run the [Gaussian Mechanism](https://saminulh.github.io/dpstats/mean_estimation/algorithms/GaussianMechanism/) [CoinPress](https://saminulh.github.io/dpstats/mean_estimation/algorithms/CoinPress/) and [PLAN](https://saminulh.github.io/dpstats/mean_estimation/algorithms/PLANBinarySearchGaussian/) on each instance at four privacy budgets, with ten seeds each. [run_sweep](https://saminulh.github.io/dpstats/library/reference/collections/dpstats.TaskCollection/#dpstats.TaskCollection.run_sweep) returns one row per trial.

In [ ]:
from dpstats.budgets import ApproxDPBudget
from dpstats.tasks import list_built_in_algorithms

built_in = list_built_in_algorithms(MeanEstimation)
algorithms = [built_in[name]() for name in ("GaussianMechanism", "CoinPress", "PLANBinarySearchGaussian")]
results = collection.run_sweep(
    algorithms=algorithms,
    instances=instances,
    budgets=[ApproxDPBudget(eps=eps, delta=1e-6) for eps in (0.3, 1.0, 3.0, 10.0)],
    seeds=range(10),
    time_limit=None,
    memory_limit_gb=None,
)
print("trials:", len(results))
results[["instance_label", "algorithm_label", "budget", "alg_seed", "status"]].head()

## Choosing a metric

`l2_error_to_sample_mean` measures the distance from each estimate to the sample mean. The metric also names its statistical baseline. [measure_results](https://saminulh.github.io/dpstats/library/reference/analysis/dpstats.analysis.measure_results/) adds a column with the metric's value to the trials. Smaller errors are better.

In [ ]:
from dpstats.analysis import measure_results
from dpstats.tasks import list_built_in_metrics

metrics = list_built_in_metrics(MeanEstimation)
metric = metrics["l2_error_to_sample_mean"]
print(metric.name, "reads", metric.required_names, "baseline", metric.baseline_name)

measurements = measure_results(results, metric, collection=collection)
measurements[["algorithm_label", "budget", "alg_seed", metric.name]].head()

## Compare with sampling error

The statistical baseline here is the standard error of the sample mean. [compute_error_ratio](https://saminulh.github.io/dpstats/library/reference/analysis/dpstats.analysis.compute_error_ratio/) divides the error of each trial by its metric's baseline: 1 is the same size as the baseline and 2 is twice as large.

The table shows the median ratio over the trials on the three instances, by privacy budget and algorithm. As the budget grows, the ratio falls, and the error from privacy becomes small next to the sampling error.

In [ ]:
from dpstats.analysis import aggregate_trials, compute_error_ratio

ratios = compute_error_ratio(measurements, metric=metric, baseline="sample_mean_l2_se")
ratios["eps"] = ratios["budget"].map(lambda budget: budget.eps)
ratios.pivot_table(
    index="eps",
    columns="algorithm_label",
    values=metric.name,
    aggfunc="median",
)

### Profile curves over trials

A profile curve shows, at each threshold $\tau$, the share of trials whose ratio is at most $\tau$, averaged over the three instances. Unlike the table, it keeps every trial instead of one median. The next cell computes the curves at $\varepsilon = 1$ with [compute_profile_curves](https://saminulh.github.io/dpstats/library/reference/analysis/dpstats.analysis.compute_profile_curves/) and plots them with [plot_profiles](https://saminulh.github.io/dpstats/library/reference/analysis/dpstats.analysis.plot_profiles/). Higher curves are better.

In [ ]:
import numpy as np

from dpstats.analysis import compute_profile_curves, plot_profiles

at_eps = ratios[ratios["eps"] == 1.0]
curves = compute_profile_curves(at_eps, np.geomspace(0.5, 30.0, 61), metric=metric)
plot_profiles(curves, xlabel="error / standard error of the sample mean").show()

## Compare with the best algorithm

A baseline can also be the best algorithm on each instance. [compute_baseline](https://saminulh.github.io/dpstats/library/reference/analysis/dpstats.analysis.compute_baseline/) takes the median error of each algorithm over the ten seeds and keeps the smallest, separately for each instance and privacy budget. Dividing by it gives a ratio where 1 means the algorithm was the best on that instance and budget.

The table shows the median ratio over the three instances.

In [ ]:
from dpstats.analysis import compute_baseline

best = compute_baseline(measurements, metric=metric, baseline=algorithms)
ratio_to_best = compute_error_ratio(measurements, best, metric=metric)
ratio_to_best["eps"] = ratio_to_best["budget"].map(lambda budget: budget.eps)
ratio_to_best.pivot_table(
    index="eps",
    columns="algorithm_label",
    values=metric.name,
    aggfunc="median",
)

### Profile curves against the best algorithm

The same profile curves, with each trial's error divided by the best algorithm's median error on its instance at $\varepsilon = 1$. A curve that reaches 1 at $\tau = 1$ never did worse than the best algorithm's median.

In [ ]:
at_eps = ratio_to_best[ratio_to_best["eps"] == 1.0]
curves = compute_profile_curves(at_eps, np.geomspace(0.3, 10.0, 61), metric=metric)
plot_profiles(curves, xlabel="error / best algorithm's median error").show()

## Summarizing trials

We compare algorithms with profile curves over trials. When one number per algorithm and instance is needed instead, [aggregate_trials](https://saminulh.github.io/dpstats/library/reference/analysis/dpstats.analysis.aggregate_trials/) takes the median over the ten seeds.

In [ ]:
from dpstats.analysis import aggregate_trials

medians = aggregate_trials(ratios, metric=metric)
medians[["instance_label", "algorithm_label", "budget", metric.name]].head()

## Define a custom metric

Define a custom [Metric](https://saminulh.github.io/dpstats/library/reference/metric/dpstats.metric.Metric/): the largest absolute coordinate distance to the sample mean. Measure the saved results again with `measure_results`, without rerunning the algorithms.

In [ ]:
import numpy as np

from dpstats.metric import Metric


def max_coordinate_error(output, *, sample_mean):
    """Largest absolute coordinate distance to the sample mean."""
    return float(np.max(np.abs(output.mean_est - np.asarray(sample_mean))))


maximum_error = Metric(
    task=MeanEstimation,
    name="max_coordinate_error",
    calculation=max_coordinate_error,
    requires=("sample_mean",),
    formula=r"\lVert \hat{\mu} - \bar{X} \rVert_\infty",
)
remeasured = measure_results(results, maximum_error, collection=collection)
remeasured[["algorithm_label", "budget", "alg_seed", maximum_error.name]].head()

## Next steps

Next: [Transformations](https://saminulh.github.io/dpstats/library/tutorials/extend_with_transformations/).